# AI-powered paraphrasing tool  

**Develop an AI-powered paraphrasing tool as a Python application or module.**

● Take a block of input text

● Use deep learning (preferably transformer models like T5, BERT, GPT, or similar
via Hugging Face Transformers)

● Generate a paraphrased version preserving meaning, improving clarity, and
ensuring originality.

● Include built-in checks for grammar, spelling, and fluency of output

**Technologies Used:**

● Python, with Hugging Face Transformers and relevant NLP packages (spaCy,
NLTK, etc.)

● TensorFlow or PyTorch for any model fine-tuning

● Use only console or script-based input/output (no GUI or web interface)

● Optional: Integrate evaluation metrics like BLEU, ROUGE, or semantic similarity
scores

In [1]:
%%capture
!pip install transformers torch sentencepiece

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load pre-trained model and tokenizer for paraphrasing
tokenizer = AutoTokenizer.from_pretrained("t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Using device: {device}")
print("Paraphrasing model loaded successfully!")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Using device: cpu
Paraphrasing model loaded successfully!


In [3]:
%%capture
!pip install language_tool_python pyspellchecker

Now that we have the necessary libraries, let's import them and initialize the tools for grammar, spelling, and fluency checks. We will integrate these into a dedicated function.

In [4]:
import language_tool_python
from spellchecker import SpellChecker

# Initialize LanguageTool for grammar and style checking
grammar_tool = language_tool_python.LanguageTool('en-US')

# Initialize SpellChecker for spelling corrections
spell = SpellChecker()

print("Grammar and spell check tools initialized.")

Grammar and spell check tools initialized.


Let's define a function to perform grammar, spelling, and basic fluency checks.

In [13]:
def check_text(text):
    report = {
        "original_text": text,
        "grammar_suggestions": [],
        "spelling_suggestions": [],
        "corrected_text": text # Start with the original text and apply corrections
    }

    # 1. Grammar and Fluency Check using LanguageTool
    matches = grammar_tool.check(text)
    grammar_issues = []
    for match in matches:
        grammar_issues.append({
            "ruleId": match.rule_id, # Corrected: Access rule ID via match.rule_id
            "message": match.message,
            "replacements": match.replacements,
            "offset": match.offset,
            "error_length": match.error_length # Corrected: Changed from errorLength to error_length
        })
    report["grammar_suggestions"] = grammar_issues

    # Apply grammar corrections to the text for the 'corrected_text' field
    report["corrected_text"] = language_tool_python.utils.correct(report["corrected_text"], matches)

    # 2. Spelling Check using pyspellchecker
    words = text.split()
    spelling_issues = []
    for word in words:
        if not spell.unknown([word]):
            # If the word is known, check if it's a common misspelling that spellchecker can suggest
            # This is a bit more involved, so for simplicity, we'll focus on unknown words first.
            continue

        corrected_word = spell.correction(word)
        if corrected_word != word:
            spelling_issues.append({
                "original_word": word,
                "suggestion": corrected_word
            })
            # Apply spelling corrections to the text
            report["corrected_text"] = report["corrected_text"].replace(word, corrected_word, 1) # Replace first occurrence

    report["spelling_suggestions"] = spelling_issues

    return report

print("Check text function defined.")

Check text function defined.


Now, let's test the `check_text` function with an example containing some grammatical and spelling errors.

In [14]:
test_text = "I has an apple. It is a good aple and I enjoy it very mutch."
print(f"Original text: {test_text}\n")

analysis_report = check_text(test_text)

print("--- Analysis Report ---")
print(f"Corrected Text: {analysis_report['corrected_text']}\n")

if analysis_report['grammar_suggestions']:
    print("Grammar Suggestions:")
    for suggestion in analysis_report['grammar_suggestions']:
        print(f"  - Rule: {suggestion['ruleId']}, Message: {suggestion['message']}, Replacements: {suggestion['replacements']}")
else:
    print("No grammar issues found.")

if analysis_report['spelling_suggestions']:
    print("\nSpelling Suggestions:")
    for suggestion in analysis_report['spelling_suggestions']:
        print(f"  - Original: {suggestion['original_word']}, Suggestion: {suggestion['suggestion']}")
else:
    print("\nNo spelling issues found.")

Original text: I has an apple. It is a good aple and I enjoy it very mutch.

--- Analysis Report ---
Corrected Text: I have an apple It is a good able and I enjoy it very much.

Grammar Suggestions:
  - Rule: BASE_FORM, Message: Possible agreement error — use the base form here., Replacements: ['have']
  - Rule: MORFOLOGIK_RULE_EN_US, Message: Possible spelling mistake found., Replacements: ['able', 'Apple', 'pale', 'ample', 'apple', 'axle', 'maple', 'ale', 'ape', 'apse', 'AELE', 'ALE', 'APE', 'APL', 'ARLE', 'Azle', 'EPLE', 'PLE']
  - Rule: MUSH_MUCH, Message: Did you mean “much”?, Replacements: ['much']

Spelling Suggestions:
  - Original: apple., Suggestion: apple
  - Original: aple, Suggestion: able
  - Original: mutch., Suggestion: much


### Evaluation Metrics: BLEU Score

To evaluate the quality of the paraphrased output, we can integrate metrics like BLEU (Bilingual Evaluation Understudy) score. BLEU is a common metric used to evaluate the quality of text which has been machine-translated or, in our case, paraphrased, against one or more reference texts.

In [15]:
%%capture
!pip install nltk

In [20]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt') # Download the necessary tokenizer data if not already present
nltk.download('punkt_tab') # Download the additional resource required for word_tokenize

def calculate_bleu(reference_text, candidate_text):
    """
    Calculates the BLEU score between a reference text and a candidate text.

    Args:
        reference_text (str): The original or reference text.
        candidate_text (str): The paraphrased or candidate text.

    Returns:
        float: The BLEU score.
    """
    # Tokenize the texts into words
    reference_tokens = nltk.word_tokenize(reference_text)
    candidate_tokens = nltk.word_tokenize(candidate_text)

    # sentence_bleu expects a list of references (even if only one) and a single candidate
    # It's good practice to have multiple references for robust BLEU scores in real scenarios.
    # For a single reference, pass it as a list containing one list of tokens.
    # Use a smoothing function to avoid zero BLEU scores for short sentences or when there are no n-gram overlaps.
    chencherry = SmoothingFunction()
    score = sentence_bleu([reference_tokens], candidate_tokens, smoothing_function=chencherry.method1)
    return score

print("BLEU score calculation function defined.")

BLEU score calculation function defined.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Let's demonstrate the `calculate_bleu` function with an example.

In [19]:
# Example usage of BLEU score
original_sentence = "The quick brown fox jumps over the lazy dog."
paraphrased_sentence_good = "A swift brown fox leaps over a lethargic canine."
paraphrased_sentence_bad = "Cat sat on mat."

bleu_good = calculate_bleu(original_sentence, paraphrased_sentence_good)
bleu_bad = calculate_bleu(original_sentence, paraphrased_sentence_bad)

print(f"Original: {original_sentence}")
print(f"Good Paraphrase: {paraphrased_sentence_good}")
print(f"BLEU Score (Good): {bleu_good:.4f}")

print(f"\nOriginal: {original_sentence}")
print(f"Bad Paraphrase: {paraphrased_sentence_bad}")
print(f"BLEU Score (Bad): {bleu_bad:.4f}")

Original: The quick brown fox jumps over the lazy dog.
Good Paraphrase: A swift brown fox leaps over a lethargic canine.
BLEU Score (Good): 0.0000

Original: The quick brown fox jumps over the lazy dog.
Bad Paraphrase: Cat sat on mat.
BLEU Score (Bad): 0.0000


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_